# Phase 4: Graph Agent (GraphSAGE/GAT) + Meta-Learner

This notebook implements:
1. **Graph Agent**: Network-based fraud detection using GraphSAGE (primary) and GAT (comparison)
2. **Meta-Learner Agent**: Ensemble model combining all agent scores

Both agents are implemented in Phase 4 as they require outputs from all previous agents.

## Setup

In [ ]:
# Kaggle-specific setup
import sys
from pathlib import Path

# Add src to path for imports
project_root = Path.cwd().parent.parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Install missing packages (Kaggle-specific)
# PyTorch Geometric requires special installation on Kaggle
!pip install -q torch-geometric==2.6.1 2>&1 | grep -v 'already satisfied' || true

In [ ]:
# Reproducibility and logging
import random
import logging
import warnings

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    print(f"PyTorch {torch.__version__}")
except ImportError:
    print("PyTorch not available â€” agents will use heuristic fallbacks")

warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO, format='%(name)s â€” %(levelname)s â€” %(message)s')
print("Setup complete.")

In [ ]:
from src.utils.device_utils import get_device, log_device_info

device_info = log_device_info()
print(f"Compute device: {device_info}")

## 1. Load Data

Inputs: canonical `transactions.parquet` and `accounts.parquet` from the interim data directory.
On Kaggle these live under `/kaggle/input/<dataset-name>/`.
A small synthetic fallback is provided for local development.

In [ ]:
%%time
from src.utils.config import get_config

cfg = get_config()

# Resolve data path: prefer Kaggle input, fall back to interim directory
KAGGLE_INPUT = Path('/kaggle/input')
if KAGGLE_INPUT.exists():
    # Adjust dataset slug as uploaded on Kaggle
    txn_path = next(KAGGLE_INPUT.rglob('transactions*.parquet'), None)
    acc_path = next(KAGGLE_INPUT.rglob('accounts*.parquet'), None)
else:
    txn_path = cfg.data_interim_dir / 'transactions.parquet'
    acc_path = cfg.data_interim_dir / 'accounts.parquet'

if txn_path and Path(txn_path).exists():
    transactions = pd.read_parquet(txn_path)
    print(f"Loaded transactions: {transactions.shape}")
else:
    # Minimal synthetic fallback for local smoke-testing
    print("No parquet data found â€” generating synthetic fallback dataset.")
    rng = np.random.default_rng(SEED)
    N = 1000
    senders = [f'ACC{rng.integers(0, 50):03d}' for _ in range(N)]
    receivers = [f'ACC{rng.integers(0, 50):03d}' for _ in range(N)]
    is_fraud = (rng.random(N) < 0.08).astype(int)
    transactions = pd.DataFrame({
        'transaction_id': [f'TXN{i:06d}' for i in range(N)],
        'sender_account_id': senders,
        'receiver_account_id': receivers,
        'amount_npr': rng.exponential(scale=50_000, size=N).clip(1, 2_000_000),
        'transaction_type': rng.choice(['transfer','payment','withdrawal','deposit'], N),
        'channel': rng.choice(['mobile_banking','atm','branch','online_banking'], N),
        'timestamp': pd.date_range('2025-01-01', periods=N, freq='min').astype(str),
        'sender_country': np.where(rng.random(N) < 0.85, 'Nepal', rng.choice(['India','Qatar','UAE','Saudi Arabia','USA'], N)),
        'receiver_country': np.where(rng.random(N) < 0.90, 'Nepal', rng.choice(['India','Qatar','UAE','Saudi Arabia','USA'], N)),
        'is_cross_border': (rng.random(N) < 0.15).astype(int),
        'original_currency': rng.choice(['NPR','USD','INR','QAR'], N, p=[0.75,0.08,0.10,0.07]),
        'ip_is_vpn': (rng.random(N) < 0.03).astype(int),
        'is_fraud': is_fraud,
        'aml_risk_indicator': is_fraud,
    })
    print(f"Synthetic fallback: {transactions.shape}, fraud rate: {is_fraud.mean():.2%}")

## 2. Graph Agent

### Architecture
- **Nodes**: bank accounts
- **Directed edges**: transactions (sender â†’ receiver), weighted by `amount_npr`
- **Node features**: out/in degree, amount sums/means, fan-in/fan-out ratios, cross-border fraction, fraud-neighbor fraction, PageRank
- **Primary model**: GraphSAGE (inductive â€” scales to unseen accounts)
- **Comparison model**: GAT (attention weights provide interpretable neighbour contributions)

### AML patterns targeted
| Pattern | Graph signature |
|---|---|
| Fan-in (collection) | High in-degree, low out-degree |
| Fan-out (distribution) | High out-degree, low in-degree |
| Layering (chains) | Long directed paths Aâ†’Bâ†’Câ†’D |
| Round-tripping | Cycles Aâ†’Bâ†’Câ†’A |
| Mule networks | Dense communities with unusual flow patterns |

In [ ]:
%%time
from src.agents.graph_agent import GraphAgent

graph_agent = GraphAgent(config={
    'graph_model_type': 'graphsage',
    'graph_hidden_channels': 64,
    'graph_epochs': 30,
    'graph_lr': 1e-3,
    'graph_dropout': 0.3,
    'graph_alert_threshold': 0.65,
    'random_seed': SEED,
})

graph_agent.fit(transactions)
print(f"GraphAgent fitted â€” {len(graph_agent._node_index)} account nodes")

In [ ]:
%%time
graph_scores = graph_agent.predict(transactions)
print(graph_scores[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']].head(10))
print(f"\nAlert rate: {graph_scores['alert_flag'].mean():.2%}")
print(f"Score range: [{graph_scores['risk_score'].min():.3f}, {graph_scores['risk_score'].max():.3f}]")

In [ ]:
# Evaluate GraphSAGE against ground truth
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

if 'is_fraud' in transactions.columns and transactions['is_fraud'].nunique() >= 2:
    y_true = transactions['is_fraud'].values
    y_score = graph_scores['risk_score'].values
    y_pred = graph_scores['alert_flag'].values

    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr = average_precision_score(y_true, y_score)
    print(f"GraphSAGE  AUC-ROC: {auc_roc:.4f}")
    print(f"GraphSAGE  AUC-PR:  {auc_pr:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=['Legitimate', 'Fraud']))
else:
    print("No ground-truth labels available for evaluation.")

In [ ]:
%%time
# GAT challenger comparison
gat_agent = GraphAgent(config={
    'graph_model_type': 'gat',
    'graph_hidden_channels': 32,
    'graph_gat_heads': 4,
    'graph_epochs': 30,
    'graph_lr': 1e-3,
    'graph_alert_threshold': 0.65,
    'random_seed': SEED,
})
gat_agent.fit(transactions)
gat_scores = gat_agent.predict(transactions)

if 'is_fraud' in transactions.columns and transactions['is_fraud'].nunique() >= 2:
    gat_auc = roc_auc_score(y_true, gat_scores['risk_score'].values)
    gat_pr  = average_precision_score(y_true, gat_scores['risk_score'].values)
    print(f"GAT        AUC-ROC: {gat_auc:.4f}")
    print(f"GAT        AUC-PR:  {gat_pr:.4f}")
    print(f"\nGraphSAGE vs GAT AUC-ROC delta: {auc_roc - gat_auc:+.4f}")

In [ ]:
# Sample explanations
sample_ids = graph_scores[graph_scores['alert_flag'] == 1]['transaction_id'].head(3).tolist()
print("--- GraphSAGE alert explanations ---")
for tid in sample_ids:
    print(graph_agent.explain(tid))

## 3. Collect All Upstream Agent Scores

The meta-learner requires scores from every upstream agent.
We run all previous agents here and join their outputs into a single wide DataFrame.

In [ ]:
%%time
from src.agents.velocity_agent import VelocityAgent
from src.agents.geo_risk_agent import GeoRiskAgent
from src.agents.behaviour_agent import BehaviourAgent
from src.agents.kyc_aml_agent import KYCAMLAgent

# Initialise all upstream agents with consistent config
upstream_agents = {
    'velocity':  VelocityAgent(config={'velocity_alert_threshold': 0.7, 'random_seed': SEED}),
    'geo_risk':  GeoRiskAgent(config={'geo_risk_alert_threshold': 0.65, 'random_seed': SEED}),
    'behaviour': BehaviourAgent(config={
        'behaviour_alert_threshold': 0.7,
        'behaviour_epochs': 5,
        'behaviour_batch_size': 64,
        'random_seed': SEED,
    }),
    'kyc_aml':   KYCAMLAgent(config={'kyc_aml_alert_threshold': 0.6}),
    'graph':     graph_agent,   # already fitted above
}

print("Upstream agents initialised.")

In [ ]:
%%time
# Fit all agents (graph already fitted)
for name, agent in upstream_agents.items():
    if not agent.is_fitted:
        print(f"  Fitting {name} agent...")
        agent.fit(transactions)

print("All upstream agents fitted.")

In [ ]:
%%time
# Run prediction for each agent and collect scores
agent_score_frames = {}
for name, agent in upstream_agents.items():
    print(f"  Scoring with {name} agent...")
    scores = agent.predict(transactions)
    agent_score_frames[name] = scores[['transaction_id', 'risk_score']].rename(
        columns={'risk_score': f'{name}_score'}
    )

# Join all agent scores into a single wide DataFrame
meta_input = transactions[['transaction_id']].copy()
if 'is_fraud' in transactions.columns:
    meta_input['is_fraud'] = transactions['is_fraud'].values

for name, frame in agent_score_frames.items():
    meta_input = meta_input.merge(frame, on='transaction_id', how='left')

# Fill any gaps where an agent returned no score
score_cols = [f'{n}_score' for n in upstream_agents]
meta_input[score_cols] = meta_input[score_cols].fillna(0.0)

print(meta_input[['transaction_id'] + score_cols].head(5).to_string())
print(f"\nMeta-input shape: {meta_input.shape}")

## 4. Meta-Learner Agent

### Design
- **Input**: one `<agent>_score` column per upstream agent
- **Primary model**: Calibrated Random Forest (isotonic regression calibration)
- **Challenger model**: XGBoost (GPU when available)
- **Calibration**: isotonic regression on held-out CV folds ensures scores are well-calibrated probabilities
- **Goal**: meta-learner AUC-ROC must exceed the best individual agent by â‰¥ 2%

In [ ]:
%%time
from src.agents.meta_learner import MetaLearnerAgent

meta_rf = MetaLearnerAgent(config={
    'meta_learner_model_type': 'random_forest',
    'meta_learner_n_estimators': 200,
    'meta_learner_calibration_method': 'isotonic',
    'meta_learner_calibration_cv': 3,
    'meta_learner_alert_threshold': 0.5,
    'random_seed': SEED,
})

meta_rf.fit(meta_input)
print("Meta-Learner (Random Forest) fitted.")

In [ ]:
%%time
meta_rf_scores = meta_rf.predict(meta_input)
print(meta_rf_scores[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']].head(10))
print(f"\nMeta-RF alert rate: {meta_rf_scores['alert_flag'].mean():.2%}")

In [ ]:
%%time
# XGBoost challenger
meta_xgb = MetaLearnerAgent(config={
    'meta_learner_model_type': 'xgboost',
    'meta_learner_n_estimators': 200,
    'meta_learner_alert_threshold': 0.5,
    'random_seed': SEED,
})
meta_xgb.fit(meta_input)
meta_xgb_scores = meta_xgb.predict(meta_input)
print("Meta-Learner (XGBoost) fitted and scored.")
print(f"XGBoost alert rate: {meta_xgb_scores['alert_flag'].mean():.2%}")

## 5. Comparative Evaluation

Comparing all agents on AUC-ROC, AUC-PR and F1 to verify the meta-learner outperforms individual agents.

In [ ]:
if 'is_fraud' in transactions.columns and transactions['is_fraud'].nunique() >= 2:
    from sklearn.metrics import f1_score

    results = []

    # Individual agent scores
    for name in upstream_agents:
        col = f'{name}_score'
        if col in meta_input.columns:
            scores_arr = meta_input[col].values
            preds = (scores_arr >= 0.5).astype(int)
            results.append({
                'Agent': name,
                'AUC-ROC': roc_auc_score(y_true, scores_arr),
                'AUC-PR':  average_precision_score(y_true, scores_arr),
                'F1':      f1_score(y_true, preds, zero_division=0),
            })

    # Meta-learner variants
    for label, scores_df in [('meta_rf', meta_rf_scores), ('meta_xgb', meta_xgb_scores)]:
        s = scores_df['risk_score'].values
        p = scores_df['alert_flag'].values
        results.append({
            'Agent': label,
            'AUC-ROC': roc_auc_score(y_true, s),
            'AUC-PR':  average_precision_score(y_true, s),
            'F1':      f1_score(y_true, p, zero_division=0),
        })

    eval_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
    eval_df = eval_df.set_index('Agent').round(4)
    print(eval_df.to_string())

    best_individual = eval_df.drop(index=['meta_rf', 'meta_xgb'], errors='ignore')['AUC-ROC'].max()
    meta_best = eval_df.loc[['meta_rf', 'meta_xgb'], 'AUC-ROC'].max()
    delta = meta_best - best_individual
    status = 'PASS' if delta >= 0.02 else 'BELOW TARGET (need â‰¥ 2% improvement)'
    print(f"\nBest individual AUC-ROC : {best_individual:.4f}")
    print(f"Best meta-learner AUC-ROC: {meta_best:.4f}")
    print(f"Delta: {delta:+.4f}  â†’  {status}")
else:
    print("No ground-truth labels available for comparative evaluation.")

## 6. Feature Importances

In [ ]:
importances = meta_rf.feature_importances
if importances:
    imp_df = (
        pd.DataFrame.from_dict(importances, orient='index', columns=['importance'])
        .sort_values('importance', ascending=False)
    )
    print("Meta-Learner (RF) feature importances:")
    print(imp_df.to_string())
else:
    print("Feature importances not available (model may be in fallback mode).")

## 7. Sample Explanations

In [ ]:
alerted = meta_rf_scores[meta_rf_scores['alert_flag'] == 1]['transaction_id'].head(5).tolist()
print("--- Meta-Learner (RF) alert explanations ---")
for tid in alerted:
    print(meta_rf.explain(tid))
    print(graph_agent.explain(tid))
    print()

## 8. Persist Fitted Agents

In [ ]:
import joblib

models_dir = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# Note: the models/ directory is git-ignored (AGENTS.md Â§10.5).
# Saved artefacts exist only in the local/Kaggle working directory and are not committed.
joblib.dump(graph_agent, models_dir / 'graph_agent_graphsage.joblib')
joblib.dump(gat_agent,   models_dir / 'graph_agent_gat.joblib')
joblib.dump(meta_rf,     models_dir / 'meta_learner_rf.joblib')
joblib.dump(meta_xgb,    models_dir / 'meta_learner_xgb.joblib')

print("Agents saved to:", models_dir)

## Conclusion

Phase 4 implemented two agents:

**Graph Agent**
- Builds a directed account-transaction graph and computes 11 structural node features
- GraphSAGE (primary) is inductive and scales to unseen accounts at inference time
- GAT (challenger) provides attention-weight interpretability for neighbour contributions
- Heuristic fallback (fan-in/fan-out ratios) when PyG is unavailable

**Meta-Learner Agent**
- Stacks all 5 upstream agent scores as features for a second-level classifier
- Calibrated Random Forest (primary) produces well-calibrated probabilities via isotonic regression
- XGBoost (challenger) with optional GPU acceleration
- Feature importances reveal which agent signals drive the combined decision

Phase 5 will add the Explanation Agent, system-level evaluation, and research documentation.